# 01 Data Cleaning

**Project:** F1 Driver vs Strategy
**Author:** Ayema Qureshi
**Stage:** Silver layer (Bronze raw CSVs in, cleaned CSVs out)

## Purpose
This notebook takes the 5 core raw CSV files from the Kaggle Formula 1 World Championship dataset (Bronze layer) and produces analysis-ready cleaned files (Silver layer). All cleaning decisions are documented inline with the reasoning behind them.

## Inputs
Raw CSVs from `data/raw/`: results, races, pit_stops, drivers, constructors

## Outputs
Cleaned CSVs written to `data/clean/`: results, races, pit_stops, drivers, constructors

## Cleaning Steps
1. Load 5 core CSVs and run sanity checks
2. Filter all files to the 2010 to 2024 analysis window using races.year, propagated through raceId, driverId, and constructorId
3. Replace `\N` text placeholders with proper nulls (pd.NA)
4. Null audit: every null classified as structural (DNFs, pre-2014 driver numbers) or scoped for column removal. The 517 pit stop "nulls" originally identified were later found to be a parse artifact, not missing data. See the pit_stops.duration section for the correction.5. Trim each DataFrame to keep-only columns per the column review
6. Rename ambiguous columns (name to race_name and constructor_name)
7. Convert data types (position and number to nullable Int64). Build duration_seconds from pit_stops.milliseconds divided by 1000, born float64 from the division, since duration strings over 60 seconds cannot be parsed by pd.to_numeric.
8. Export cleaned DataFrames to `data/clean/`

## Key Decisions
- No rows dropped and no values imputed. All nulls are structural and carry meaning.
- Pit stop data starts in 2011, so 2010 is a known gap for pit stop features.
- Nullable Int64 used to preserve whole-number columns that contain structural nulls.
- CSV export does not preserve nullable dtype metadata, so downstream notebooks must re-cast on load.

In [2]:
import pandas as pd

In [3]:
# Load in core files into dataframes
results = pd.read_csv('../data/raw/results.csv')
races = pd.read_csv('../data/raw/races.csv')
pit_stops = pd.read_csv('../data/raw/pit_stops.csv')
drivers = pd.read_csv('../data/raw/drivers.csv')  
constructors = pd.read_csv('../data/raw/constructors.csv')

## Sanity Check: Core File Loading

All 5 core CSV files loaded successfully from `data/raw/`.

| File | Rows | Columns | Notes |
|------|------|---------|-------|
| results | 26,759 | 18 | Row count matches data audit. All columns appear non-null because `\N` is stored as text. `position`, `number`, `time`, `milliseconds`, `fastestLap`, `rank`, `fastestLapTime`, `fastestLapSpeed` are read as string type and will need conversion after `\N` replacement. |
| races | 1,125 | 18 | `\N` visible in `head()` for practice session, qualifying, and sprint columns. `year` is int64, ready for filtering. |
| pit_stops | 11,371 | 7 | `duration` loaded as string, needs float conversion. Data starts 2011 not 2010. |
| drivers | 861 | 9 | `number` and `code` are strings, will contain `\N` for pre-2014 drivers. |
| constructors | 212 | 5 | Cleanest file. No issues detected. |

**Key finding:** Every column across all 5 files shows 0 nulls. This is misleading. The dataset uses `\N` as a text placeholder instead of true null values, so pandas reads them as valid strings. Once `\N` is replaced with proper nulls, null counts will change significantly, especially `position` in results.csv which should be approximately 40% null due to DNFs.

**Next step:** Filter to 2010 to 2024 using `races.year`, then replace `\N` text values with proper nulls.

In [4]:
races.columns

Index(['raceId', 'year', 'round', 'circuitId', 'name', 'date', 'time', 'url',
       'fp1_date', 'fp1_time', 'fp2_date', 'fp2_time', 'fp3_date', 'fp3_time',
       'quali_date', 'quali_time', 'sprint_date', 'sprint_time'],
      dtype='object')

In [5]:
# Filter races to the 2010 to 2024 analysis window
races = races[(races['year'] >= 2010) & (races['year'] <= 2024)].copy()

# Build the list of valid race IDs from the filtered races
valid_race_ids = races['raceId']

# Keep only rows in the other files that belong to those races
results = results[results['raceId'].isin(valid_race_ids)].copy()
pit_stops = pit_stops[pit_stops['raceId'].isin(valid_race_ids)].copy()

# Trim the lookup tables to only drivers and constructors who appear in the window
drivers = drivers[drivers['driverId'].isin(results['driverId'])].copy()
constructors = constructors[constructors['constructorId'].isin(results['constructorId'])].copy()

# Sanity check row counts after filtering
print(f"races: {races.shape}")
print(f"results: {results.shape}")
print(f"pit_stops: {pit_stops.shape}")
print(f"drivers: {drivers.shape}")
print(f"constructors: {constructors.shape}")

races: (305, 18)
results: (6436, 18)
pit_stops: (11371, 7)
drivers: (80, 9)
constructors: (23, 5)


## Filtering to the 2010 to 2024 Analysis Window

Filtered all 5 core files to only include races from 2010 to 2024. Since only `races.csv` has a `year` column, the filter flows through `raceId`:

1. Filter `races` on `year` between 2010 and 2024
2. Extract valid `raceId` values from the filtered `races`
3. Filter `results` and `pit_stops` using `.isin()` on `raceId`
4. Filter `drivers` and `constructors` using `.isin()` on the driver and constructor IDs found in the filtered `results`

`.copy()` is applied after each filter to create independent DataFrames, preventing `SettingWithCopyWarning` when modifying columns in later steps.

### Row Counts After Filtering

| File | Rows | Columns |
|------|------|---------|
| races | 305 | 18 |
| results | 6,436 | 18 |
| pit_stops | 11,371 | 7 |
| drivers | 80 | 9 |
| constructors | 23 | 5 |

Row counts are consistent with expectations: 305 races across 15 seasons (about 20 per year), 6,436 results at roughly 20 drivers per race, pit_stops unchanged since the data was already 2011 onward, and drivers and constructors reduced to only those active in this window.

In [6]:
# Replace \N with proper nulls across all 5 DataFrames, write a loop

# Create a dictionary of the dataframes to iterate over
dataframes = {
    'races': races,
    'results': results,
    'pit_stops': pit_stops,
    'drivers': drivers,
    'constructors': constructors
}

# Loop through the dictionary and replace \N with pd.NA in each dataframe
for name, df in dataframes.items():
    dataframes[name] = df.replace('\\N', pd.NA)

races = dataframes['races']
results = dataframes['results']
pit_stops = dataframes['pit_stops']
drivers = dataframes['drivers']
constructors = dataframes['constructors']

In [7]:
# Loop through the dictionary and print out null summary for each dataframe
for name, df in dataframes.items():
    print(f"\n{'='*50}")
    print(f"{name} null summary:")
    print(f"{'='*50}")
    print(df.isnull().sum())


races null summary:
raceId           0
year             0
round            0
circuitId        0
name             0
date             0
time             0
url              0
fp1_date       215
fp1_time       237
fp2_date       215
fp2_time       237
fp3_date       233
fp3_time       252
quali_date     215
quali_time     237
sprint_date    287
sprint_time    290
dtype: int64

results null summary:
resultId              0
raceId                0
driverId              0
constructorId         0
number                0
grid                  0
position           1039
positionText          0
positionOrder         0
points                0
laps                  0
time               3126
milliseconds       3126
fastestLap          285
rank                 27
fastestLapTime      285
fastestLapSpeed     285
statusId              0
dtype: int64

pit_stops null summary:
raceId          0
driverId        0
stop            0
lap             0
time            0
duration        0
milliseconds    0
dtype

## Null Audit After Replacing `\N`

After replacing text `\N` values with proper nulls, the null landscape breaks into three categories: structural nulls that carry meaning, nulls in columns that are already scoped for removal, and clean files with no nulls.

### races
All 10 columns with nulls (fp1_date, fp1_time, fp2_date, fp2_time, fp3_date, fp3_time, quali_date, quali_time, sprint_date, sprint_time) are already scoped as skip columns. Nulls will be resolved when these columns are dropped in the next step.

### results
| Column | Nulls | Decision | Reason |
|--------|-------|----------|--------|
| position | 1,039 | Leave as null | Structural. Represents DNFs (did not finish). Will be excluded from `position_change` calculation and documented as a modeling limitation. |
| time, milliseconds | 3,126 each | Drop with column | Already scoped as skip. Race time is only clean for the winner, everyone else has a relative gap. |
| fastestLap, fastestLapTime, fastestLapSpeed | 285 each | Drop with column | Already scoped as skip. |
| rank | 27 | Drop with column | Already scoped as skip. |

### pit_stops
No nulls. Clean file.

### drivers
`number` has 21 nulls. This reflects a real-world F1 rule change: permanent driver numbers were only introduced in 2014, so drivers who raced only before 2014 have no permanent number. Decision: leave as null. This is a structural null tied to a documented rule change, not a data quality issue.

### constructors
No nulls. Clean file.

### Summary
No rows will be dropped for null handling. No values will be imputed. Structural nulls are preserved because they carry information (DNFs, pre-2014 driver numbers). All other nulls are in columns already scoped for removal.

In [8]:
# Create a dictionary of the dataframes to use only keep columns that are needed for the analysis
keep_columns = {
    'results': ['raceId', 'driverId', 'constructorId', 'grid', 'position', 'positionText', 'points', 'statusId'],
    'races': ['raceId', 'year', 'circuitId', 'name'], 
    'pit_stops': ['raceId', 'driverId', 'stop', 'lap', 'duration', 'milliseconds'],
    'drivers': ['driverId', 'driverRef', 'number', 'code', 'forename', 'surname', 'nationality'],
    'constructors': ['constructorId', 'constructorRef', 'name', 'nationality']
}

In [9]:
# Rename columns for clarity and consistency across dataframes
rename_map = {
    'races': {'name': 'race_name'},
    'constructors': {'name': 'constructor_name'}
}

In [10]:
# For each entry in keep_columns, grab the name and the column list. Go to the matching DataFrame in dataframes, filter it to those columns, and save the copy back
for name, cols in keep_columns.items():
    dataframes[name] = dataframes[name][cols].copy()

In [11]:
races = dataframes['races']
results = dataframes['results']
pit_stops = dataframes['pit_stops']
drivers = dataframes['drivers']
constructors = dataframes['constructors']

In [12]:
print(f"races: {races.shape}")
print(f"results: {results.shape}")
print(f"pit_stops: {pit_stops.shape}")
print(f"drivers: {drivers.shape}")
print(f"constructors: {constructors.shape}")

races: (305, 4)
results: (6436, 8)
pit_stops: (11371, 6)
drivers: (80, 7)
constructors: (23, 4)


In [13]:
# Create a for loop for rename columns
for names, mapping in rename_map.items():
    dataframes[names] = dataframes[names].rename(columns=mapping)

In [14]:
races = dataframes['races']
constructors = dataframes['constructors']

## Trimming to Keep-Only Columns and Renaming for Clarity

After replacing `\N` text values with `pd.NA`, the next step was reducing each DataFrame to only the columns needed for the analysis and renaming ambiguous columns for clarity.

### Approach

Two production patterns used:

**1. Trim step: dictionary-and-loop.** Built a `keep_columns` dictionary mapping each DataFrame name to its list of columns to retain. Looped through the dictionary and filtered each DataFrame down to only those columns, using `.copy()` to create independent objects. This pattern is more maintainable than 5 separate filtering lines because adding or removing a column later means updating one dictionary entry, not hunting through scattered code.

**2. Rename step: separate cell, separate operation.** Built a `rename_map` dictionary with only the DataFrames that needed renames (races and constructors). Kept trim and rename as separate cells because they are two distinct jobs. Combining them would reduce readability during code review.

### Rename Decisions

Both `races.csv` and `constructors.csv` originally have a column named `name`. Left as-is, joining these tables would create ambiguous column collisions. Renamed for clarity:

| File | Original | Renamed To |
|------|----------|------------|
| races | name | race_name |
| constructors | name | constructor_name |

`circuits.csv` also has a `name` column that will be renamed to `circuit_name` when that file is brought in later.

### Column Counts After Trim

| File | Columns Before | Columns After |
|------|---------------|---------------|
| results | 18 | 8 |
| races | 18 | 4 |
| pit_stops | 7 | 5 |
| drivers | 9 | 7 |
| constructors | 5 | 4 |

All trim decisions align with the column-by-column review documented in the project brief. No new column decisions were made during this step.

### Next Step

Fix data types on string columns that should be numeric, prioritizing `results.position` and `pit_stops.duration` since these feed the engineered features `position_change` and `avg_pit_stop_per_team_per_season`.

In [15]:
# Results - position should be numeric (integer with null allowed)
results['position'] = pd.to_numeric(results['position'], errors='coerce')

# Pit stops - duration strings over 60 seconds are stored as MM:SS.mmm and cannot be parsed 
# by pd.to_numeric. Build duration_seconds from milliseconds instead, which holds every value.
# The original duration string column stays as human-readable evidence.
pit_stops['duration_seconds'] = pit_stops['milliseconds'] / 1000

# Driver - number is stored as string. Since it has structural nulls from pre-2014 drivers, 
# it should be a nullable integer type.
drivers['number'] = pd.to_numeric(drivers['number'], errors='coerce')

In [16]:
# Build a dictionary of the dataframes for astype for production and pipeline use
dtypes_dict = {
    'results': {
        'position': 'Int64'
    },
    'drivers': {
        'number': 'Int64'
    }
}

In [17]:
# Loop through dtypes_dict. For each DataFrame, pass its column-to-type mapping 
# directly to .astype(). No nested loop needed because .astype() accepts a dict 
# and handles the per-column conversion internally.
for name, cols in dtypes_dict.items():
    dataframes[name] = dataframes[name].astype(cols)

In [18]:
results = dataframes['results']
drivers = dataframes['drivers']
pit_stops = dataframes['pit_stops']

## Fixing Data Types

After trimming to keep-only columns and applying renames, three columns still had incorrect data types. Fixing them now is important because two of these columns feed directly into engineered features that require numeric operations.

### Approach

Each column was reviewed against what it actually represents in the real world, and the target type was chosen to match that intent. No blanket conversion was applied.

**1. `results.position`.** Represents a driver's finishing position in a race. Was stored as a string in the raw CSV because it originally contained a mix of numeric values ("1", "2", "3") and `\N` text for drivers who did not finish. Feeds directly into the `position_change` engineered feature (grid minus finish), which requires subtraction. Converted to nullable integer (`Int64`) because finishing positions are whole numbers and DNF nulls must be preserved.

**2. `pit_stops.duration`.** Represents how many seconds a pit stop took. Was stored as a string in the raw CSV. This is the primary team strategy proxy for the entire project, so it must support averaging, correlation, and comparison. Converted to float (`float64`) because pit stop times have decimal precision that matters (a 22.5 second stop is meaningfully different from a 22.9 second stop).

**3. `drivers.number`.** Represents a driver's permanent car number. Was stored as a string in the raw CSV. Converted to nullable integer (`Int64`) because car numbers are whole numbers and the pre-2014 structural nulls must be preserved.

### Why `Int64` Instead of `int64`

Regular pandas integer (`int64`, lowercase) does not allow nulls. If a column contains even one null, pandas silently downgrades the entire column to `float64` so the null has somewhere to live. This is why running `pd.to_numeric()` on `position` and `number` originally returned floats (values like `1.0` and `44.0`) instead of clean whole numbers.

Nullable integer (`Int64`, capital I) is pandas' newer type that allows both whole numbers and nulls in the same column. Using it preserves the intent of these columns (whole numbers) while accommodating the structural nulls documented in the null audit.

### The `errors='coerce'` Argument

`pd.to_numeric()` was called with `errors='coerce'` on all three columns. This tells pandas: if a value cannot be converted to a number, turn it into a null instead of raising an error. This is the correct choice here because the columns already contain `pd.NA` values from the earlier `\N` replacement step, and those must be preserved as nulls, not cause the operation to fail.

### Context: The Pre-2014 Driver Number Rule

The structural nulls in `drivers.number` are not a data quality issue. Before the 2014 season, F1 drivers did not have permanent career numbers. A driver's number changed race to race based on their team's championship position from the previous season. Permanent driver numbers were only introduced in 2014, when drivers selected a number to use for the rest of their career (for example, Lewis Hamilton chose 44, Max Verstappen chose 33).

Because the Kaggle dataset only records permanent numbers, any driver who raced exclusively before 2014 has no number to record, and the field was stored as `\N` in the raw CSV. These nulls are structural, expected, and documented as a limitation. They do not affect analysis because joins across tables use `driverId`, not `number`.

### Column Types After Conversion

| Column | Before | After | Reason |
|---|---|---|---|
| `results.position` | string | Int64 | Whole numbers, DNF nulls preserved |
| `pit_stops.duration` | string | float64 | Decimal precision matters |
| `drivers.number` | string | Int64 | Whole numbers, pre-2014 nulls preserved |

### Null Accounting for `results.position`

Every one of the 1039 nulls in `results.position` is explained by `positionText`:

| Code | Count | Meaning |
|---|---|---|
| R | 979 | Retired (mechanical, crash, DNF) |
| W | 37 | Withdrew before race |
| D | 13 | Disqualified after race |
| N | 5 | Not classified (finished under 90% of winner's distance) |
| F | 4 | Failed to qualify |
| E | 1 | Excluded |

No nulls are unexplained. No imputation needed. This is why `positionText` was kept alongside `position` during the trim step.

## Investigating Nulls in pit_stops.duration

In [19]:
pit_stops.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11371 entries, 0 to 11370
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   raceId            11371 non-null  int64  
 1   driverId          11371 non-null  int64  
 2   stop              11371 non-null  int64  
 3   lap               11371 non-null  int64  
 4   duration          11371 non-null  object 
 5   milliseconds      11371 non-null  int64  
 6   duration_seconds  11371 non-null  float64
dtypes: float64(1), int64(5), object(1)
memory usage: 622.0+ KB


### Duration Null Investigation

After the type conversion, `pit_stops.duration` had 517 nulls. These were not accounted for in the original null audit, so an investigation was needed to determine whether they represented a data quality issue, a structural pattern, or something else.

In [20]:
pit_stops[pit_stops['duration'].isna()].merge(races[['raceId', 'year']], on='raceId')['year'].value_counts().sort_index()

Series([], Name: count, dtype: int64)

Nulls do not follow a "data quality improving over time" pattern. Recent years like 2021 (110) and 2023 (104) have the most nulls, while early years like 2011 and 2013 are nearly clean. This suggests nulls are event-driven, not systemic. Next: check specific races.

In [21]:
nulls_by_race = pit_stops[pit_stops['duration'].isna()].merge(
    races[['raceId', 'year', 'race_name']], on='raceId'
)
nulls_by_race.groupby(['year', 'race_name']).size().sort_values(ascending=False).head(20)

Series([], dtype: int64)

## Null Accounting for `pit_stops.duration`

Original conclusion (July 3, 2026): the 517 rows failing `pd.to_numeric` conversion were interpreted as missing values from red flag races where duration was not captured. That conclusion was wrong. The values were never missing.

### What the 517 rows actually contain
On July 6, 2026, an external review flagged that the numeric conversion itself created the 517 nulls: the source data reported 0 nulls in `duration` before conversion, and 517 after. Investigating what the failing strings actually looked like revealed the mechanism.

Pit stops of 60 seconds or longer are stored in the raw CSV as `MM:SS.mmm` strings, for example `16:44.718` and `2:03.124`. `pd.to_numeric` cannot parse a colon and stamped these rows NaN, silently. The values existed all along in a format the converter could not read.

The `milliseconds` column, originally scoped out of the cleaning pipeline, holds every one of these values: 0 nulls among the 517, and `milliseconds / 1000` matches `duration` exactly on all normal rows. This makes milliseconds the authoritative source of truth for pit stop duration.

### What these long stops represent
Once decoded via milliseconds, the failing rows show a median duration of roughly 22 minutes and a maximum of roughly 51 minutes. These are race stoppage events, cars parked in pit lane during red flags, not crew service. The concentration at known red flag races (2023 Australian GP, 2016 Brazilian GP, 2021 Saudi Arabian GP, 2020 Tuscan GP, 2021 Emilia Romagna GP) confirms this.

### Correction applied to the cleaning pipeline
1. `milliseconds` is now retained in `keep_columns` for `pit_stops`.
2. The `pd.to_numeric` line on `duration` was removed. It was the exact line manufacturing the false nulls.
3. A new numeric column `duration_seconds` is built as `pit_stops['milliseconds'] / 1000`. This column is born float64 and contains every stop, including the 60+ second ones.
4. The original `duration` string column stays in place as human-readable evidence.
5. The `pit_stops` entry in `dtypes_dict` was removed because `duration_seconds` is born float64 from the division.

### Impact on Feature 5
The 60+ second stops are not crew service events and cannot be averaged in as if they were. Including them would shift team-season averages by up to 225 seconds and destroy the crew execution proxy. They will be excluded from `avg_pit_stop_per_team_per_season` by a deliberate documented rule in notebook 02 (`duration_seconds < 60`), not by an accidental parse failure. The exclusion is now honest, verified, and defensible.

### Handling
No imputation. No row removal. `duration_seconds` is fully populated with 0 nulls, and the crew-service exclusion is enforced downstream in feature 5 with an explicit rule.

### Note in the AI use log
External AI review (ChatGPT, July 6, 2026) correctly diagnosed the mechanism behind the 517 nulls. Its proposed remedy was to include the long stops in Feature 5, which was rejected on evidence: including them shifts averages by up to 225 seconds and poisons the crew execution proxy. Diagnosis accepted, remedy rejected with data. Logged as a Discernment example under the four D's framework.

In [22]:
# export the cleaned dataframes to CSV files in the processed folder by creating a for loop and data dict
dataframes_to_export = {
    'results': results,
    'races': races,
    'pit_stops': pit_stops,
    'drivers': drivers,
    'constructors': constructors
}   

In [23]:
# Loop through the dictionary and export each dataframe to a CSV file in the processed folder
for name, df in dataframes_to_export.items():
    df.to_csv(f'../data/clean/{name}.csv', index=False)